# 05 - ClinicalBERT on masked text (seeds 42, 43, 44)

**Owner:** Gabriel Barrere (masked runs). Yuhan runs the same script on the original text.
**Depends on:** `src/data.py`, `src/masking.py`, `src/train_clinicalbert.py` (Yuhan's, called unchanged), `src/evaluate.py`
**Spec:** `docs/clinicalbert_pipeline.md` (PR #28). The only difference from the original runs is `--masked`.

What this notebook does:
1. Sets up Colab (Drive, repo, data, packages) and saves `pip freeze` for pinning `requirements.txt`.
2. Runs counts-only checks (no note text is printed anywhere).
3. Trains the 3 masked seeds by calling Yuhan's script. Checkpoints go to Google Drive only.
4. Validates the 3 validation prediction files and evaluates them.

Rules: **the test split is never loaded here** (`load_split_notes` refuses it). Never commit note text, the Kaggle key or checkpoints. Clear all outputs before committing this notebook.

Before running: **Runtime > Change runtime type > T4 GPU**. Add `KAGGLE_USERNAME`, `KAGGLE_KEY` and `GITHUB_TOKEN` (read-only token for this private repo) under the key icon (Secrets). About 25 to 30 min per seed.

## 0. Setup

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "github.com/gab-architech/exsu500-clinical-nlp.git"
BRANCH = "gabriel-masked-runs"

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    ROOT = Path("/content/exsu500-clinical-nlp")
    if not ROOT.exists():
        token = userdata.get("GITHUB_TOKEN")
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, f"https://{token}@{REPO_URL}", str(ROOT)], check=True)
        # Remove the token from the saved remote URL so it never ends up in logs or outputs
        subprocess.run(["git", "remote", "set-url", "origin", f"https://{REPO_URL}"], cwd=ROOT, check=True)
    DRIVE_DIR = Path("/content/drive/MyDrive/exsu500")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE_DIR = Path.home() / "exsu500_runs"  # local stand-in for Drive, outside the repo

COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
sys.path.insert(0, str(ROOT / "src"))
print(f"Running in {'Colab' if IN_COLAB else 'local'}; repo {ROOT} at commit {COMMIT}")

## 1. CONFIG

In [ ]:
RANDOM_SEED = 42                 # project-wide seed (split, bootstrap)
SEEDS = [42, 43, 44]             # one training run per seed, same as Yuhan's original runs

CKPT_DIR = DRIVE_DIR / "checkpoints"          # Drive only, never the repo
RESULTS_DIR = DRIVE_DIR / "results_masked"    # predictions, run logs, metrics
FIG_DIR = DRIVE_DIR / "figures_masked"
ENV_DIR = DRIVE_DIR / "env"
for d in [CKPT_DIR, RESULTS_DIR, FIG_DIR, ENV_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 2. Data and packages

In [ ]:
# Download mtsamples.csv into data/ (data/README.md, option 2). The key stays in Colab Secrets.
import os

if not (ROOT / "data" / "mtsamples.csv").exists():
    if not IN_COLAB:
        raise FileNotFoundError("Download mtsamples.csv into data/ first (data/README.md, option 1)")
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    subprocess.run(["kaggle", "datasets", "download", "-d", "tboyle10/medicaltranscriptions",
                    "-p", str(ROOT / "data"), "--unzip", "-q"], check=True)
print("data file present:", (ROOT / "data" / "mtsamples.csv").exists())

In [ ]:
# Install the project requirements, then save the exact versions (spec item 7: pin from the first real run)
if IN_COLAB:  # locally, use your own environment
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(ENV_DIR / f"pip_freeze_{COMMIT}.txt").write_text(freeze)
print(f"saved {len(freeze.splitlines())} package versions to {ENV_DIR}")

## 3. Checks (counts only)

Expected: train 1,560 and val 335; under masking option C, 159 train and 35 val notes change (241 of 2,230 overall, 47 of them in test, which is not loaded here).

In [ ]:
import torch
from data import load_split_notes
from masking import contains_masked_term

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU would take hours)")

train_df = load_split_notes("train")
val_df = load_split_notes("val")
n_train_masked = int(train_df["text"].map(contains_masked_term).sum())
n_val_masked = int(val_df["text"].map(contains_masked_term).sum())
print(f"train {len(train_df)}, val {len(val_df)}; notes changed by masking: train {n_train_masked}, val {n_val_masked}")
assert (len(train_df), len(val_df)) == (1560, 335)
assert (n_train_masked, n_val_masked) == (159, 35), "masking list differs from the agreed option C"

In [ ]:
# Recompute "tokens kept" with the agreed 255 + 255 head+tail (docs/midterm_numbers.md said 59.7%, all notes).
# Train + val only, because this notebook never reads test. Original (unmasked) text, as in the midterm figure.
import numpy as np
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
lengths = np.array([len(ids) for ids in tok(list(train_df["text"]) + list(val_df["text"]),
                                            add_special_tokens=False)["input_ids"]])
kept = np.minimum(lengths, 510)
print(f"notes: {len(lengths)}, median length {np.median(lengths):.0f} tokens, "
      f"over 510: {(lengths > 510).mean():.1%}, tokens kept: {kept.sum() / lengths.sum():.1%}")

## 4. Train the masked runs

Calls Yuhan's script unchanged with `--masked`. A seed whose prediction file already exists on Drive is skipped, so after a Colab disconnect just re-run sections 0 to 1 and this cell.

In [ ]:
for seed in SEEDS:
    pred_file = RESULTS_DIR / f"preds_clinicalbert_masked_{seed}_val.csv"
    if pred_file.exists():
        print(f"seed {seed}: already done, skipping")
        continue
    print(f"seed {seed}: training")
    subprocess.run([sys.executable, "src/train_clinicalbert.py", "--seed", str(seed), "--masked",
                    "--output-dir", str(CKPT_DIR), "--results-dir", str(RESULTS_DIR)],
                   cwd=ROOT, check=True)
    assert pred_file.exists(), f"seed {seed}: no prediction file written"

## 5. Validate and evaluate (validation split only)

In [ ]:
pred_files = sorted(str(p) for p in RESULTS_DIR.glob("preds_clinicalbert_masked_*_val.csv"))
subprocess.run([sys.executable, "src/validate_predictions.py", *pred_files], cwd=ROOT, check=True)
subprocess.run([sys.executable, "src/evaluate.py", "--split", "val", "--pred-dir", str(RESULTS_DIR),
                "--out-dir", str(RESULTS_DIR), "--fig-dir", str(FIG_DIR)], cwd=ROOT, check=True)
print((RESULTS_DIR / "summary_val.md").read_text())

In [ ]:
# Run logs: one line per seed, and a check that all seeds used the same code and library versions
import pandas as pd

logs = [json.loads((RESULTS_DIR / "run_logs" / f"clinicalbert_masked_seed{s}.json").read_text()) for s in SEEDS]
print(pd.DataFrame([{"seed": g["seed"], "val_macro_f1": round(g["validation_macro_f1"], 4),
                     "minutes": round(g["wall_clock_seconds"] / 60, 1), "gpu": g["gpu"], "commit": g["commit"]}
                    for g in logs]).to_string(index=False))
assert len({g["commit"] for g in logs}) == 1, "seeds ran on different commits"
assert len({json.dumps(g["versions"], sort_keys=True) for g in logs}) == 1, "seeds ran on different library versions"

## 6. Hand-off (manual)

1. From Drive `exsu500/results_masked/`, download the 3 `preds_clinicalbert_masked_*_val.csv` files and `run_logs/`, and copy them into the repo's `results/` on your machine (they hold note IDs, labels and probabilities only, no text).
2. From `exsu500/env/`, use the freeze file to pin `requirements.txt` (the key libraries only), and agree it with Yuhan.
3. Checkpoints stay in `exsu500/checkpoints/` on Drive. Test predictions come from them after the Nov 25 model lock.
4. **Edit > Clear all outputs** before saving this notebook to the repo.